# Computação Gráfica – Projeto 1
Criação de uma cena de sala de estar utilizando objetos compostos a partir de formas básicas.

Feito por:
    * 
    *

Lista dos controles:

> * **[G]:**                            para ligar e desligar o ventilador.
> * **[W], [A], [S], [D]:**             para controlar o robô aspirador.
> * **[J], [K]:**                       para abrir e fechar a cortina.
> * **[C]:**                            para mudar a cor da tela da tv.
> * **[↑], [↓], [→], [←], [Z], [X]:**   para controlar a rotação da cena.
> * **[R]:**                            para reiniciar a cena.
> * **[P]:**                            para mostrar a malha poligonal.

### Instalação de dependências

In [156]:
import glfw
from OpenGL.GL import *
import OpenGL.GL.shaders
import numpy as np
import glm
import math
import ctypes
import random

### Inicializando janela

In [157]:
glfw.init()
glfw.window_hint(glfw.VISIBLE, glfw.FALSE)
window = glfw.create_window(1350, 1050, "Programa", None, None)

if (window == None):
    print("Failed to create GLFW window")
    glfw.terminate()

glfw.make_context_current(window)

----
## Shaders

Note que agora usamos vec3, já que estamos em 3D.

In [158]:
vertex_code = """
        attribute vec3 position;
        uniform mat4 mat_transformation;
        void main(){
            gl_Position = mat_transformation * vec4(position,1.0);
        }
        """

In [159]:
fragment_code = """
        uniform vec4 color;
        void main(){
            gl_FragColor = color;
        }
        """

### Requisitando slot para a GPU para nossos programas Vertex e Fragment Shaders

In [160]:
# Request a program and shader slots from GPU
program  = glCreateProgram()
vertex   = glCreateShader(GL_VERTEX_SHADER)
fragment = glCreateShader(GL_FRAGMENT_SHADER)

### Associando nosso código-fonte aos slots solicitados

In [161]:
# Set shaders source
glShaderSource(vertex, vertex_code)
glShaderSource(fragment, fragment_code)

### Compilando o Vertex Shader

Se há algum erro em nosso programa Vertex Shader, nosso app para por aqui.

In [162]:
# Compile shaders
glCompileShader(vertex)
if not glGetShaderiv(vertex, GL_COMPILE_STATUS):
    error = glGetShaderInfoLog(vertex).decode()
    print(error)
    raise RuntimeError("Erro de compilacao do Vertex Shader")

### Compilando o Fragment Shader

Se há algum erro em nosso programa Fragment Shader, nosso app para por aqui.

In [163]:
glCompileShader(fragment)
if not glGetShaderiv(fragment, GL_COMPILE_STATUS):
    error = glGetShaderInfoLog(fragment).decode()
    print(error)
    raise RuntimeError("Erro de compilacao do Fragment Shader")

### Associando os programas compilados ao programa principal

In [164]:
# Attach shader objects to the program
glAttachShader(program, vertex)
glAttachShader(program, fragment)

### Linkagem do programa

In [165]:
# Build program
glLinkProgram(program)
if not glGetProgramiv(program, GL_LINK_STATUS):
    print(glGetProgramInfoLog(program))
    raise RuntimeError('Linking error')

# Make program the default program
glUseProgram(program)

### Preparando dados para enviar a GPU

Agora vamos modelar algumas formas 3D básicas.


----------------------------------------------------------
## Modelagem das formas básicas 

Modelagem das primitivas usadas para compor os objetos finais futuramente

### Funções auxiliares para modelagem

In [166]:
PI = math.pi

def adiciona_triangulo(vertices, p0, p1, p2):
    vertices.append(p0)
    vertices.append(p1)
    vertices.append(p2)

def adiciona_quad(vertices, p0, p1, p2, p3):
    vertices.append(p0)
    vertices.append(p1)
    vertices.append(p2)

    vertices.append(p0)
    vertices.append(p2)
    vertices.append(p3)

### Modelando um cubo

In [167]:
def cria_cubo(tamanho=1.0):
    h = tamanho / 2.0
    vertices = []

    p000 = (-h, -h, -h)
    p001 = (-h, -h,  h)
    p010 = (-h,  h, -h)
    p011 = (-h,  h,  h)
    p100 = ( h, -h, -h)
    p101 = ( h, -h,  h)
    p110 = ( h,  h, -h)
    p111 = ( h,  h,  h)

    adiciona_quad(vertices, p001, p101, p111, p011)
    adiciona_quad(vertices, p000, p010, p110, p100)
    adiciona_quad(vertices, p000, p001, p011, p010)
    adiciona_quad(vertices, p100, p110, p111, p101)
    adiciona_quad(vertices, p010, p011, p111, p110)
    adiciona_quad(vertices, p000, p100, p101, p001)

    return vertices

### Modelando uma pirâmide de base quadrada

In [168]:
def cria_piramide(base=1.0, altura=1.2):
    hb = base / 2.0
    z_base = -altura / 2.0
    z_topo = altura / 2.0

    vertices = []

    p0 = (-hb, -hb, z_base)
    p1 = ( hb, -hb, z_base)
    p2 = ( hb,  hb, z_base)
    p3 = (-hb,  hb, z_base)
    topo = (0.0, 0.0, z_topo)

    adiciona_quad(vertices, p0, p1, p2, p3)
    adiciona_triangulo(vertices, p0, p1, topo)
    adiciona_triangulo(vertices, p1, p2, topo)
    adiciona_triangulo(vertices, p2, p3, topo)
    adiciona_triangulo(vertices, p3, p0, topo)

    return vertices

### Modelando um prisma triangular

In [169]:
def cria_prisma_triangular(largura=1.0, altura=1.0, comprimento=1.2):
    hx = largura / 2.0
    hy = altura / 2.0
    hz = comprimento / 2.0

    vertices = []

    a0 = (-hx, -hy,  hz)
    a1 = ( hx, -hy,  hz)
    a2 = (0.0,  hy,  hz)

    b0 = (-hx, -hy, -hz)
    b1 = ( hx, -hy, -hz)
    b2 = (0.0,  hy, -hz)

    adiciona_triangulo(vertices, a0, a1, a2)
    adiciona_triangulo(vertices, b0, b2, b1)

    adiciona_quad(vertices, a0, b0, b1, a1)
    adiciona_quad(vertices, a1, b1, b2, a2)
    adiciona_quad(vertices, a2, b2, b0, a0)

    return vertices

### Modelando um cilindro

In [170]:
def cria_cilindro(raio=0.5, altura=1.2, setores=24):
    vertices = []
    z0 = -altura / 2.0
    z1 = altura / 2.0
    passo = (2.0 * PI) / setores

    centro_base = (0.0, 0.0, z0)
    centro_topo = (0.0, 0.0, z1)

    for i in range(setores):
        a0 = i * passo
        a1 = (i + 1) * passo

        p0 = (raio * math.cos(a0), raio * math.sin(a0), z0)
        p1 = (raio * math.cos(a1), raio * math.sin(a1), z0)
        p2 = (raio * math.cos(a0), raio * math.sin(a0), z1)
        p3 = (raio * math.cos(a1), raio * math.sin(a1), z1)

        adiciona_triangulo(vertices, centro_base, p1, p0)
        adiciona_triangulo(vertices, centro_topo, p2, p3)

        vertices.append(p0)
        vertices.append(p1)
        vertices.append(p3)

        vertices.append(p0)
        vertices.append(p3)
        vertices.append(p2)

    return vertices

### Modelando um cone

In [171]:
def cria_cone(raio=0.5, altura=1.2, setores=24):
    vertices = []
    z_base = -altura / 2.0
    z_topo = altura / 2.0
    passo = (2.0 * PI) / setores

    centro_base = (0.0, 0.0, z_base)
    topo = (0.0, 0.0, z_topo)

    for i in range(setores):
        a0 = i * passo
        a1 = (i + 1) * passo

        p0 = (raio * math.cos(a0), raio * math.sin(a0), z_base)
        p1 = (raio * math.cos(a1), raio * math.sin(a1), z_base)

        adiciona_triangulo(vertices, centro_base, p1, p0)
        adiciona_triangulo(vertices, p0, p1, topo)

    return vertices

### Modelando uma esfera

In [172]:
def cria_esfera(raio=0.5, setores=24, stacks=16):
    vertices = []

    for i in range(stacks):
        phi0 = -PI / 2.0 + i * PI / stacks
        phi1 = -PI / 2.0 + (i + 1) * PI / stacks

        for j in range(setores):
            theta0 = j * 2.0 * PI / setores
            theta1 = (j + 1) * 2.0 * PI / setores

            p0 = (raio * math.cos(phi0) * math.cos(theta0),
                  raio * math.cos(phi0) * math.sin(theta0),
                  raio * math.sin(phi0))

            p1 = (raio * math.cos(phi0) * math.cos(theta1),
                  raio * math.cos(phi0) * math.sin(theta1),
                  raio * math.sin(phi0))

            p2 = (raio * math.cos(phi1) * math.cos(theta1),
                  raio * math.cos(phi1) * math.sin(theta1),
                  raio * math.sin(phi1))

            p3 = (raio * math.cos(phi1) * math.cos(theta0),
                  raio * math.cos(phi1) * math.sin(theta0),
                  raio * math.sin(phi1))

            adiciona_triangulo(vertices, p0, p1, p2)
            adiciona_triangulo(vertices, p0, p2, p3)

    return vertices

### Chamada das formas básicas

In [173]:
vertices_list = []

cubo = cria_cubo()
inicio_cubo = len(vertices_list)
vertices_list.extend(cubo)
quantos_vertices_cubo = len(cubo)

piramide = cria_piramide()
inicio_piramide = len(vertices_list)
vertices_list.extend(piramide)
quantos_vertices_piramide = len(piramide)

prisma = cria_prisma_triangular()
inicio_prisma = len(vertices_list)
vertices_list.extend(prisma)
quantos_vertices_prisma = len(prisma)

cilindro = cria_cilindro()
inicio_cilindro = len(vertices_list)
vertices_list.extend(cilindro)
quantos_vertices_cilindro = len(cilindro)

cone = cria_cone()
inicio_cone = len(vertices_list)
vertices_list.extend(cone)
quantos_vertices_cone = len(cone)

esfera = cria_esfera()
inicio_esfera = len(vertices_list)
vertices_list.extend(esfera)
quantos_vertices_esfera = len(esfera)

### Empacotando os vértices

In [174]:
vertices = np.zeros(len(vertices_list), [("position", np.float32, 3)])
vertices["position"] = vertices_list

----------------
## Envio ao buffer e alocação na GPU

### Para enviar nossos dados da CPU para a GPU, precisamos requisitar um slot (buffer).

In [175]:
# Request a buffer slot from GPU
buffer_VBO = glGenBuffers(1)
# Make this buffer the default one
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO)

### Abaixo, nós enviamos todo o conteúdo da variável vertices.

In [176]:
# Upload data
glBufferData(GL_ARRAY_BUFFER, vertices.nbytes, vertices, GL_DYNAMIC_DRAW)
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO)

### Associando variáveis do programa GLSL (Vertex Shader) com nossos dados

In [177]:
# Bind the position attribute
stride = vertices.strides[0]
offset = ctypes.c_void_p(0)

Em seguida, soliciamos à GPU a localização da variável "position" (que guarda coordenadas dos nossos vértices). Nós definimos essa variável no Vertex Shader.

In [178]:
loc = glGetAttribLocation(program, "position")
glEnableVertexAttribArray(loc)

A partir da localização anterior, nós indicamos à GPU onde está o conteúdo (via posições stride/offset) para a variável position (aqui identificada na posição loc).

Outros parâmetros:

* Definimos que possui <b> três </b> coordenadas
* Que cada coordenada é do tipo float (GL_FLOAT)
* Que não se deve normalizar a coordenada (False)

Mais detalhes: https://www.khronos.org/registry/OpenGL-Refpages/gl4/html/glVertexAttribPointer.xhtml

In [179]:
glVertexAttribPointer(loc, 3, GL_FLOAT, False, stride, offset)

### Vamos pegar a localização da variável color para definir a cor de cada forma

In [180]:
loc_color = glGetUniformLocation(program, "color")

---
## Eventos de teclado

In [181]:
malha_poligonal = False

# Variaveis globais do ventilador
angulo_ventilador_x = 0.0
angulo_ventilador_y = 0.0
angulo_ventilador_z = 0.0
angulo_laminas = 0.0
ventilador_ligado = False

#Variaveis globais do sofa
angulo_sofa_x = 0.0
angulo_sofa_y = 0.0
angulo_sofa_z = 0.0

# Variaveis globais do aspirador
angulo_aspirador_x = 0.0
angulo_aspirador_y = 0.0
angulo_aspirador_z = 0.0
desl_aspirador_x = 0.0
desl_aspirador_y = 0.0
desl_aspirador_z = 0.0

# Variaveis globais do ambiente
angulo_ambiente_x = 0.0
angulo_ambiente_y = 0.0
angulo_ambiente_z = 0.0

# Variaveis globais da cortina
fechamento_cortina = 1.0
angulo_cortina_x = 0.0
angulo_cortina_y = 0.0
angulo_cortina_z = 0.0


# Variaveis globais da janela
angulo_janela_x = 0
angulo_janela_y = 0
angulo_janela_z = 0

# Variaveis globais da tv
angulo_tv_x = 0.0
angulo_tv_y = 0.0
angulo_tv_z = 0.0
cor_tv_r = 0.0
cor_tv_g = 0.0
cor_tv_b = 0.0

angulo_cena_x = -35
angulo_cena_y = -40
angulo_cena_z = 24

'''
angulo_cena_x = -35
angulo_cena_y = -40
angulo_cena_z = 24

angulo_cena_x = 0
angulo_cena_y = -90
angulo_cena_z = 0
'''

def key_event(window, key, scancode, action, mods):
    '''
    Grande parte das variáveis globais estão aqui para facilitar a manipulação dos objetos,
    já que a função de desenho é chamada a todo momento e não é possível passar parâmetros para ela durante a execução.
    Assim, as variáveis globais são modificadas na função de evento de teclado e o valores obtidos foram posteriormente
    utilizados na chamada função de desenho. Por esta razão, encontram-se comentadas as manipulações individuais de cada objeto.
    '''
    global malha_poligonal, angulo_ventilador_x, angulo_ventilador_y, angulo_ventilador_z, angulo_ventilador_z, ventilador_ligado, angulo_sofa_x, angulo_sofa_y, angulo_sofa_z, angulo_aspirador_x, angulo_aspirador_y  , angulo_aspirador_z, angulo_ambiente_x, angulo_ambiente_y, angulo_ambiente_z, angulo_cena_x, angulo_cena_y, angulo_cena_z, desl_aspirador_x, desl_aspirador_y, desl_aspirador_z, angulo_tv_x, angulo_tv_y, angulo_tv_z, cor_tv_r, cor_tv_g, cor_tv_b, fechamento_cortina, angulo_cortina_x, angulo_cortina_y, angulo_cortina_z, angulo_janela_x, angulo_janela_y, angulo_janela_z 


    v_rotação = 5.0
    v_aspirador = 0.05

    if action == glfw.PRESS or action == glfw.REPEAT:

        if key == glfw.KEY_P:
            malha_poligonal = not malha_poligonal

        if key == glfw.KEY_R:
            angulo_cena_x = -35
            angulo_cena_y = -40
            angulo_cena_z = 24

            desl_aspirador_x = 0.0
            desl_aspirador_y = 0.0
            desl_aspirador_z = 0.0

            cor_tv_r = 0.0
            cor_tv_g = 0.0
            cor_tv_b = 0.0

            ventilador_ligado = False

            fechamento_cortina = 1.0

        if key == glfw.KEY_LEFT:
            '''angulo_ventilador_y += v_rotação
            angulo_sofa_y += v_rotação
            angulo_aspirador_y += v_rotação
            angulo_ambiente_y += v_rotação'''
            angulo_cena_y += v_rotação

        if key == glfw.KEY_RIGHT:
            '''angulo_ventilador_y -= v_rotação
            angulo_sofa_y -= v_rotação
            angulo_aspirador_y -= v_rotação
            angulo_ambiente_y -= v_rotação'''
            angulo_cena_y -= v_rotação

        if key == glfw.KEY_UP:
            '''angulo_ventilador_x += v_rotação
            angulo_sofa_x += v_rotação
            angulo_aspirador_x += v_rotação
            angulo_ambiente_x += v_rotação'''
            angulo_cena_x += v_rotação

        if key == glfw.KEY_DOWN:
            '''angulo_ventilador_x -= v_rotação
            angulo_sofa_x -= v_rotação
            angulo_aspirador_x -= v_rotação
            angulo_ambiente_x -= v_rotação'''
            angulo_cena_x -= v_rotação

        if key == glfw.KEY_Z:
            '''angulo_ambiente_z += v_rotação'''
            angulo_cena_z += v_rotação

        if key == glfw.KEY_X:
            '''angulo_ambiente_z -= v_rotação'''
            angulo_cena_z -= v_rotação

        if key == glfw.KEY_W:
            desl_aspirador_x += v_aspirador

        if key == glfw.KEY_S:
            desl_aspirador_x -= v_aspirador

        if key == glfw.KEY_A:
            desl_aspirador_z += v_aspirador

        if key == glfw.KEY_D:
            desl_aspirador_z -= v_aspirador

        if key == glfw.KEY_C:
            cor_tv_r = random.uniform(0.0, 1.0)
            cor_tv_g = random.uniform(0.0, 1.0)
            cor_tv_b = random.uniform(0.0, 1.0)
            if cor_tv_r > 1.0:
                cor_tv_r = 0.0
            if cor_tv_g > 1.0:
                cor_tv_g = 0.0
            if cor_tv_b > 1.0:
                cor_tv_b = 0.0

        if key == glfw.KEY_J:
            if fechamento_cortina < 1.0:
                fechamento_cortina += 0.02
        
        if key == glfw.KEY_K:
            if fechamento_cortina > 0.02:
                fechamento_cortina -= 0.02

        if key == glfw.KEY_G:
            ventilador_ligado = not ventilador_ligado

glfw.set_key_callback(window, key_event)


---
## Matrizes de transformação

In [182]:
def multiplica_matriz(a, b):
    m_a = a.reshape(4,4)
    m_b = b.reshape(4,4)
    m_c = np.dot(m_a, m_b)
    c = m_c.reshape(1,16)
    return c


def matriz_rotacao_x(angulo):
    angulo = math.radians(angulo)
    cos_d = math.cos(angulo)
    sin_d = math.sin(angulo)
    return np.array([1.0,   0.0,    0.0, 0.0,
                     0.0, cos_d, -sin_d, 0.0,
                     0.0, sin_d,  cos_d, 0.0,
                     0.0,   0.0,    0.0, 1.0], np.float32)


def matriz_rotacao_y(angulo):
    angulo = math.radians(angulo)
    cos_d = math.cos(angulo)
    sin_d = math.sin(angulo)
    return np.array([ cos_d, 0.0, sin_d, 0.0,
                      0.0,   1.0,  0.0, 0.0,
                     -sin_d, 0.0, cos_d, 0.0,
                      0.0,   0.0,  0.0, 1.0], np.float32)


def matriz_rotacao_z(angulo):
    angulo = math.radians(angulo)
    cos_d = math.cos(angulo)
    sin_d = math.sin(angulo)
    return np.array([cos_d, -sin_d, 0.0, 0.0,
                     sin_d,  cos_d, 0.0, 0.0,
                     0.0,    0.0,   1.0, 0.0,
                     0.0,    0.0,   0.0, 1.0], np.float32)


def matriz_translacao(t_x, t_y, t_z):
    return np.array([1.0, 0.0, 0.0, t_x,
                     0.0, 1.0, 0.0, t_y,
                     0.0, 0.0, 1.0, t_z,
                     0.0, 0.0, 0.0, 1.0], np.float32)


def matriz_escala(s_x, s_y, s_z):
    return np.array([s_x, 0.0, 0.0, 0.0,
                     0.0, s_y, 0.0, 0.0,
                     0.0, 0.0, s_z, 0.0,
                     0.0, 0.0, 0.0, 1.0], np.float32)


---
## Funções de desenho dos objetos finais

Os objetos finais têm como objetivo compor a cena de uma sala de estar. Por esse motivo a lista completa de objetos contempla:

1. **Ventilador**: Que pode ser ligado e desligado apertando a tecla *[G]*. Composto por 3 cilindros e 4 prismas. ***(ROTAÇÃO)***
2. **Sofá**: Composto por 12 cubos.
3. **Aspirador**: Que pode se movimentar usando W, A, S, D. Composto por 7 cilindros. ***(TRANSLAÇÃO)***
4. **Ambiente**: Composto por 3 cubos.
5. **Televisão**: Composta por 2 cubos. A tecla *[C]* muda a cor da tela. 
6. **Cortina**: Formada por 1 cilindro e 11 cubos. "Fecha" e "Abre" com *[J]* e *[K]*. ***(ESCALA)***
7. **Janela**: Formada por 7 cubos.

A lógica por trás da criação de cada objeto é constante em cada função, seguindo o seguinte padrão:
* Primeiro, lê-se da chamada variáveis de posição, escala e rotação (do objeto como um todo) e as armazena (elas serão usadas na matriz de transformação do objeto).
* Depois, cria-se matrizes de transformação para cada parte individual do objeto (cada primitiva, como cubos, cilindros, etc). Cronologicamente falando, a codificação de cada uma dessas funções começou por aqui.
* Então, parte a parte, primeiro escala-se, depois rotaciona-se e, por fim, traslada-se.
* Aplica-se daí as transformações globais do objeto (como mudar a escala do ventilador como um todo).
* E, finalmente, aplica-se uma transformação opcional usando o matriz da cena, passada como argumento da função que é declarada no loop principal da cena. Nesse caso em particular, essa matriz foi usada para pordermos rotacionar a cena como um todo e pordemos vê-la a partir de ângulos variádos.

Ou seja, o padrão de transformações para cada parte do objeto segue o esquema:

> Transformações de parte --> Transformações globais de objeto --> Transformações de cena

In [183]:
# Disclaimer: Durante o desenvolvimento, o ângulo dos objetos pôde ser alterado a partir de eventos do teclado para facilitar o posicionamento na cena


'''
Função de criação e posicionamento do ventilador.
Parametros:
    - pos_x, pos_y, pos_z: posição do ventilador no espaço
    - escala_ventilador: escala do ventilador como um todo
    - angulo_x, angulo_y, angulo_z: ângulo de rotação do ventilador em torno dos eixos x, y e z

Aperte "G" para ligar/desligar a rotação das lâminas do ventilador! 
'''
def desenha_ventilador(pos_x, pos_y, pos_z, escala_ventilador, angulo_x, angulo_y, angulo_z, mat_cena):

    loc = glGetUniformLocation(program, "mat_transformation")


    # Esse trecho mexe na posição do ventilador como um todo ----------------------------------------
    centro_x = pos_x
    centro_y = pos_y
    centro_z = pos_z
    angulo_ventilador_x 
    angulo_ventilador_y
    angulo_ventilador_z

    mat_rot_global_x = matriz_rotacao_x(angulo_ventilador_x + angulo_x)
    mat_rot_global_y = matriz_rotacao_y(angulo_ventilador_y + angulo_y)
    mat_rot_global_z = matriz_rotacao_z(angulo_ventilador_z + angulo_z)
    mat_scale_global = matriz_escala(escala_ventilador, escala_ventilador, escala_ventilador)
    mat_translation_global = matriz_translacao(centro_x, centro_y, centro_z)

    # BASE -------------------------------------------------------------------------------------------
    mat_translation = matriz_translacao(0.0, -0.30, 0.0)
    mat_rotation_x = matriz_rotacao_x(90)
    mat_scale = matriz_escala(0.48, 0.25, 0.022)

    mat_transform = multiplica_matriz(mat_rotation_x, mat_scale)
    mat_transform = multiplica_matriz(mat_translation, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)

    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, 0.32, 0.32, 0.34, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_prisma, quantos_vertices_prisma)

    # TRONCO -------------------------------------------------------------------------------------------
    mat_translation = matriz_translacao(0.0, 0.0, 0.0)
    mat_rotation_x = matriz_rotacao_x(90)
    mat_scale = matriz_escala(0.1, 0.1, 0.5)

    mat_transform = multiplica_matriz(mat_rotation_x, mat_scale)
    mat_transform = multiplica_matriz(mat_translation, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)

    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, 0.55, 0.55, 0.57, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_cilindro, quantos_vertices_cilindro)

    # PRATO -------------------------------------------------------------------------------------------
    mat_translation = matriz_translacao(0.0, 0.40, -0.05)
    mat_scale = matriz_escala(0.5, 0.5, 0.03)

    mat_transform = multiplica_matriz(mat_translation, mat_scale)
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)

    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, 0.68, 0.68, 0.70, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_cilindro, quantos_vertices_cilindro)

    # EIXO -------------------------------------------------------------------------------------------
    mat_translation = matriz_translacao(0.0, 0.40, -0.07)
    mat_scale = matriz_escala(0.05, 0.05, 0.08)

    mat_transform = multiplica_matriz(mat_translation, mat_scale)
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)

    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, 0.22, 0.22, 0.24, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_cilindro, quantos_vertices_cilindro)

    # LÂMINA 1 -------------------------------------------------------------------------------------------
    mat_translation = matriz_translacao(0.0, 0.40, -0.08)
    mat_rotation_z = matriz_rotacao_z(60 + angulo_laminas)
    mat_scale = matriz_escala(0.45, 0.06, 0.012)

    mat_transform = multiplica_matriz(mat_rotation_z, mat_scale)
    mat_transform = multiplica_matriz(mat_translation, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)

    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, 0.8, 0.8, 0.8, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_prisma, quantos_vertices_prisma)

    # LÂMINA 2 -------------------------------------------------------------------------------------------
    mat_translation = matriz_translacao(0.0, 0.40, -0.08)
    mat_rotation_z = matriz_rotacao_z(0 + angulo_laminas)
    mat_scale = matriz_escala(0.45, 0.06, 0.012)

    mat_transform = multiplica_matriz(mat_rotation_z, mat_scale)
    mat_transform = multiplica_matriz(mat_translation, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)

    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, 0.8, 0.8, 0.8, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_prisma, quantos_vertices_prisma)

    # LÂMINA 3 -------------------------------------------------------------------------------------------
    mat_translation = matriz_translacao(0.0, 0.40, -0.08)
    mat_rotation_z = matriz_rotacao_z(-60 + angulo_laminas)
    mat_scale = matriz_escala(0.45, 0.06, 0.012)

    mat_transform = multiplica_matriz(mat_rotation_z, mat_scale)
    mat_transform = multiplica_matriz(mat_translation, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)

    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, 0.8, 0.8, 0.8, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_prisma, quantos_vertices_prisma)



'''
Função de criação e posicionamento do sofa.
Parametros:
    - pos_x, pos_y, pos_z: posição do sofa no espaço
    - escala_sofa: escala do sofa como um todo
    - angulo_x, angulo_y, angulo_z: ângulo de rotação do sofa em torno dos eixos x, y e z
'''
def desenha_sofa(pos_x, pos_y, pos_z, escala_sofa, angulo_x, angulo_y, angulo_z, mat_cena):

    loc = glGetUniformLocation(program, "mat_transformation")

    # Esse trecho mexe na posição do sofa como um todo ----------------------------------------------
    centro_x = pos_x
    centro_y = pos_y
    centro_z = pos_z
    angulo_sofa_x
    angulo_sofa_y
    angulo_sofa_z

    mat_rot_global_x = matriz_rotacao_x(angulo_sofa_x + angulo_x)
    mat_rot_global_y = matriz_rotacao_y(angulo_sofa_y + angulo_y)
    mat_rot_global_z = matriz_rotacao_z(angulo_sofa_z + angulo_z)
    mat_scale_global = matriz_escala(escala_sofa, escala_sofa, escala_sofa)
    mat_translation_global = matriz_translacao(centro_x, centro_y, centro_z)

    # BASE -------------------------------------------------------------------------------------------
    mat_translation = matriz_translacao(0.0, -0.18, 0.0)
    mat_scale = matriz_escala(0.95, 0.18, 0.32)

    mat_transform = multiplica_matriz(mat_translation, mat_scale)
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)

    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, 0.72, 0.67, 0.58, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_cubo, quantos_vertices_cubo)


    # BRAÇO 1 -------------------------------------------------------------------------------------------
    mat_translation = matriz_translacao(-0.40, 0.08, 0.0)
    mat_scale = matriz_escala(0.15, 0.34, 0.32)

    mat_transform = multiplica_matriz(mat_translation, mat_scale)
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)

    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, 0.76, 0.71, 0.62, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_cubo, quantos_vertices_cubo)


    # BRAÇO 2 -------------------------------------------------------------------------------------------
    mat_translation = matriz_translacao(0.40, 0.08, 0.0)
    mat_scale = matriz_escala(0.15, 0.34, 0.32)

    mat_transform = multiplica_matriz(mat_translation, mat_scale)
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)

    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, 0.76, 0.71, 0.62, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_cubo, quantos_vertices_cubo)


    # ASSENTO 1 -------------------------------------------------------------------------------------------
    mat_translation = matriz_translacao(-0.17, 0.00, 0.0)
    mat_scale = matriz_escala(0.31, 0.12, 0.26)

    mat_transform = multiplica_matriz(mat_translation, mat_scale)
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)

    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, 0.82, 0.78, 0.70, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_cubo, quantos_vertices_cubo)


    # ASSENTO 2 -------------------------------------------------------------------------------------------
    mat_translation = matriz_translacao(0.17, 0.00, 0.0)
    mat_scale = matriz_escala(0.31, 0.12, 0.26)

    mat_transform = multiplica_matriz(mat_translation, mat_scale)
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)

    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, 0.82, 0.78, 0.70, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_cubo, quantos_vertices_cubo)


    # TAMPA DE TRÁS -------------------------------------------------------------------------------------------
    mat_translation = matriz_translacao(0.0, -0.01, 0.20)
    mat_scale = matriz_escala(0.95, 0.52, 0.08)

    mat_transform = multiplica_matriz(mat_translation, mat_scale)
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)

    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, 0.74, 0.71, 0.60, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_cubo, quantos_vertices_cubo)


    # ALMOFADA 1 -------------------------------------------------------------------------------------------
    mat_translation = matriz_translacao(-0.16, 0.18, 0.15)
    mat_scale = matriz_escala(0.28, 0.22, 0.06)

    mat_transform = multiplica_matriz(mat_translation, mat_scale)
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)

    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, 0.86, 0.81, 0.73, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_cubo, quantos_vertices_cubo)


    # ALMOFADA 2 -------------------------------------------------------------------------------------------
    mat_translation = matriz_translacao(0.16, 0.18, 0.15)
    mat_scale = matriz_escala(0.28, 0.22, 0.06)

    mat_transform = multiplica_matriz(mat_translation, mat_scale)
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)

    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, 0.86, 0.81, 0.73, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_cubo, quantos_vertices_cubo)


    # PÉ 1 -------------------------------------------------------------------------------------------
    mat_translation = matriz_translacao(-0.38, -0.30, -0.12)
    mat_scale = matriz_escala(0.05, 0.08, 0.05)

    mat_transform = multiplica_matriz(mat_translation, mat_scale)
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)

    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, 0.22, 0.18, 0.14, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_cubo, quantos_vertices_cubo)


    # PÉ 2 -------------------------------------------------------------------------------------------
    mat_translation = matriz_translacao(0.38, -0.30, -0.12)
    mat_scale = matriz_escala(0.05, 0.08, 0.05)

    mat_transform = multiplica_matriz(mat_translation, mat_scale)
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)

    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, 0.22, 0.18, 0.14, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_cubo, quantos_vertices_cubo)


    # PÉ 3 -------------------------------------------------------------------------------------------
    mat_translation = matriz_translacao(-0.38, -0.30, 0.12)
    mat_scale = matriz_escala(0.05, 0.08, 0.05)

    mat_transform = multiplica_matriz(mat_translation, mat_scale)
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)

    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, 0.22, 0.18, 0.14, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_cubo, quantos_vertices_cubo)


    # PÉ 4 -------------------------------------------------------------------------------------------
    mat_translation = matriz_translacao(0.38, -0.30, 0.12)
    mat_scale = matriz_escala(0.05, 0.08, 0.05)

    mat_transform = multiplica_matriz(mat_translation, mat_scale)
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)

    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, 0.22, 0.18, 0.14, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_cubo, quantos_vertices_cubo)



'''
Função de criação e posicionamento do aspirador robô.
Parametros:
    - pos_x, pos_y, pos_z: posição do aspirador no espaço
    - escala_aspirador: escala do aspirador como um todo
    - angulo_x, angulo_y, angulo_z: ângulo de rotação do aspirador em torno dos eixos x, y e z

Use W,A,S,D para movimentar o aspirador!
'''
def desenha_aspirador(pos_x, pos_y, pos_z, escala_aspirador, angulo_x, angulo_y, angulo_z, mat_cena):

    loc = glGetUniformLocation(program, "mat_transformation")

    # Esse trecho mexe na posição do aspirador como um todo ----------------------------------------
    centro_x = pos_x
    centro_y = pos_y
    centro_z = pos_z
    angulo_aspirador_x
    angulo_aspirador_y
    angulo_aspirador_z

    mat_rot_global_x = matriz_rotacao_x(angulo_aspirador_x + angulo_x)
    mat_rot_global_y = matriz_rotacao_y(angulo_aspirador_y + angulo_y)
    mat_rot_global_z = matriz_rotacao_z(angulo_aspirador_z + angulo_z)
    mat_scale_global = matriz_escala(escala_aspirador, escala_aspirador, escala_aspirador)
    mat_translation_global = matriz_translacao(centro_x + desl_aspirador_x, centro_y + desl_aspirador_y, centro_z + desl_aspirador_z)

    # DISCO EXTERNO -------------------------------------------------------------------------------------------
    mat_translation = matriz_translacao(0.0, 0.0, 0.0)
    mat_rotation_x = matriz_rotacao_x(0)
    mat_scale = matriz_escala(0.55, 0.55, 0.08)

    mat_transform = multiplica_matriz(mat_rotation_x, mat_scale)
    mat_transform = multiplica_matriz(mat_translation, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)

    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, 0.86, 0.86, 0.86, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_cilindro, quantos_vertices_cilindro)

    # DISCO INTERNO -------------------------------------------------------------------------------------------
    mat_translation = matriz_translacao(0.0, 0.0, 0.03)
    mat_scale = matriz_escala(0.46, 0.46, 0.07)

    mat_transform = multiplica_matriz(mat_translation, mat_scale)
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)

    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, 0.60, 0.60, 0.60, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_cilindro, quantos_vertices_cilindro)

    # SENSOR -------------------------------------------------------------------------------------------
    mat_translation = matriz_translacao(0.0, 0.13, 0.07)
    mat_scale = matriz_escala(0.10, 0.10, 0.06)

    mat_transform = multiplica_matriz(mat_translation, mat_scale)
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)

    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, 0.30, 0.30, 0.30, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_cilindro, quantos_vertices_cilindro)

    # MOP 1 -------------------------------------------------------------------------------------------
    mat_translation = matriz_translacao(-0.12, 0.18, -0.06)
    mat_scale = matriz_escala(0.26, 0.26, 0.03)

    mat_transform = multiplica_matriz(mat_translation, mat_scale)
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)

    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, 0.95, 0.95, 0.90, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_cilindro, quantos_vertices_cilindro)

    # MOP 2 -------------------------------------------------------------------------------------------
    mat_translation = matriz_translacao(0.12, 0.18, -0.06)
    mat_scale = matriz_escala(0.26, 0.26, 0.03)

    mat_transform = multiplica_matriz(mat_translation, mat_scale)
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)

    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, 0.95, 0.95, 0.90, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_cilindro, quantos_vertices_cilindro)

    # RODA 1 -------------------------------------------------------------------------------------------
    mat_translation = matriz_translacao(-0.18, -0.04, -0.04)
    mat_rotation_y = matriz_rotacao_y(90)
    mat_scale = matriz_escala(0.09, 0.09, 0.03)

    mat_transform = multiplica_matriz(mat_rotation_y, mat_scale)
    mat_transform = multiplica_matriz(mat_translation, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)

    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, 0.12, 0.12, 0.12, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_cilindro, quantos_vertices_cilindro)

    # RODA 2 -------------------------------------------------------------------------------------------
    mat_translation = matriz_translacao(0.18, -0.04, -0.04)
    mat_rotation_y = matriz_rotacao_y(90)
    mat_scale = matriz_escala(0.09, 0.09, 0.03)

    mat_transform = multiplica_matriz(mat_rotation_y, mat_scale)
    mat_transform = multiplica_matriz(mat_translation, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)

    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, 0.12, 0.12, 0.12, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_cilindro, quantos_vertices_cilindro)



def desenha_ambiente(pos_x, pos_y, pos_z, escala_ambiente, angulo_x, angulo_y, angulo_z, mat_cena):

    loc = glGetUniformLocation(program, "mat_transformation")

    # Esse trecho mexe na posição do ambiente como um todo ----------------------------------------
    centro_x = pos_x
    centro_y = pos_y
    centro_z = pos_z
    angulo_ambiente_x
    angulo_ambiente_y
    angulo_ambiente_z

    mat_rot_global_x = matriz_rotacao_x(angulo_ambiente_x + angulo_x)
    mat_rot_global_y = matriz_rotacao_y(angulo_ambiente_y + angulo_y)
    mat_rot_global_z = matriz_rotacao_z(angulo_ambiente_z + angulo_z)
    mat_scale_global = matriz_escala(escala_ambiente, escala_ambiente, escala_ambiente)
    mat_translation_global = matriz_translacao(centro_x, centro_y, centro_z)

    # CHÃO -------------------------------------------------------------------------------------------
    mat_translation = matriz_translacao(0.0, -0.59, 0.0)
    mat_rotation_x = matriz_rotacao_x(0)
    mat_scale = matriz_escala(2.40, 0.08, 2.40)

    mat_transform = multiplica_matriz(mat_rotation_x, mat_scale)
    mat_transform = multiplica_matriz(mat_translation, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)

    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, 0.5, 0.35, 0.25, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_cubo, quantos_vertices_cubo)

    # PAREDE DE TRÁS -------------------------------------------------------------------------------------------
    mat_translation = matriz_translacao(0.0, 0.45, 1.16)
    mat_rotation_x = matriz_rotacao_x(0)
    mat_scale = matriz_escala(2.40, 2.00, 0.08)

    mat_transform = multiplica_matriz(mat_rotation_x, mat_scale)
    mat_transform = multiplica_matriz(mat_translation, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)

    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, 0.90, 0.70, 0.01, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_cubo, quantos_vertices_cubo)

    # PAREDE LATERAL -------------------------------------------------------------------------------------------
    mat_translation = matriz_translacao(1.16, 0.45, 0.0)
    mat_rotation_x = matriz_rotacao_x(0)
    mat_scale = matriz_escala(0.08, 2.00, 2.40)

    mat_transform = multiplica_matriz(mat_rotation_x, mat_scale)
    mat_transform = multiplica_matriz(mat_translation, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)

    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, 0.80, 0.55, 0.01, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_cubo, quantos_vertices_cubo)



'''
Função de criação e posicionamento da televisão.
Parametros:
    - pos_x, pos_y, pos_z: posição da televisão no espaço
    - escala_tv: escala da televisão como um todo
    - angulo_x, angulo_y, angulo_z: ângulo de rotação da televisão em torno dos eixos x, y e z
'''
def desenha_tv(pos_x, pos_y, pos_z, escala_tv, angulo_x, angulo_y, angulo_z, mat_cena):

    loc = glGetUniformLocation(program, "mat_transformation")

    cor_tv_r
    cor_tv_g
    cor_tv_b

    # Esse trecho mexe na posição da televisão como um todo ----------------------------------------
    centro_x = pos_x
    centro_y = pos_y
    centro_z = pos_z
    angulo_tv_x
    angulo_tv_y
    angulo_tv_z

    mat_rot_global_x = matriz_rotacao_x(angulo_tv_x + angulo_x)
    mat_rot_global_y = matriz_rotacao_y(angulo_tv_y + angulo_y)
    mat_rot_global_z = matriz_rotacao_z(angulo_tv_z + angulo_z)
    mat_scale_global = matriz_escala(escala_tv, escala_tv, escala_tv)
    mat_translation_global = matriz_translacao(centro_x, centro_y, centro_z)

    # CORPO -------------------------------------------------------------------------------------------
    mat_translation = matriz_translacao(0.0, 0.0, 0.0)
    mat_scale = matriz_escala(0.60, 0.34, 0.05)

    mat_transform = multiplica_matriz(mat_translation, mat_scale)
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)

    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, 0.08, 0.08, 0.08, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_cubo, quantos_vertices_cubo)

    # TELA -------------------------------------------------------------------------------------------
    mat_translation = matriz_translacao(0.0, 0.0, -0.025)
    mat_scale = matriz_escala(0.52, 0.27, 0.01)

    mat_transform = multiplica_matriz(mat_translation, mat_scale)
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)

    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, cor_tv_r, cor_tv_g, cor_tv_b, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_cubo, quantos_vertices_cubo)


'''
Função de criação e posicionamento da cortina.
Parametros:
    - pos_x, pos_y, pos_z: posição da cortina no espaço
    - escala_cotina: escala da cortina como um todo
    - angulo_x, angulo_y, angulo_z: ângulo de rotação da cortina em torno dos eixos x, y e z
    - fechamento_cortina: escala a largura do tecido da cortina para fechar e abrir ela, ajustando sua posição

Use J para fechar a cortina e K para abrir!
'''
def desenha_cortina(pos_x, pos_y, pos_z, escala_cortina, angulo_x, angulo_y, angulo_z, mat_cena):
    
    loc = glGetUniformLocation(program, "mat_transformation")

     # Esse trecho mexe na posição da cortina como um todo
    centro_x = pos_x
    centro_y = pos_y
    centro_z = pos_z

    angulo_cortina_x
    angulo_cortina_y
    angulo_cortina_z

    mat_rot_global_x = matriz_rotacao_x(angulo_cortina_x + angulo_x)
    mat_rot_global_y = matriz_rotacao_y(angulo_cortina_y + angulo_y)
    mat_rot_global_z = matriz_rotacao_z(angulo_cortina_z + angulo_z)
    mat_scale_global = matriz_escala(escala_cortina, escala_cortina, escala_cortina)
    mat_translation_global = matriz_translacao(centro_x, centro_y, centro_z)

    # MADEIRA -----------------------------------------------------
    mat_translation = matriz_translacao(0.0, 0.5, 0.0)
    mat_rotation_y = matriz_rotacao_y(90)
    mat_scale = matriz_escala(0.05, 0.05, 1.2)

    mat_transform = multiplica_matriz(mat_rotation_y, mat_scale)
    mat_transform = multiplica_matriz(mat_translation, mat_transform)
    
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)


    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, 0.3, 0.2, 0.1, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_cilindro, quantos_vertices_cilindro)

    # TECIDOS -------------------------------------------------
    posicoes_x = [0.6, 0.45, 0.3, 0.15, 0.0, -0.15, -0.3, -0.45, -0.6]
    
    for i, px in enumerate(posicoes_x):
        angulo_tira = 45 if i % 2 == 0 else -45
        
        if i % 2 == 0:
            glUniform4f(loc_color, 0.6, 0.7, 0.9, 1.0)
        else:
            glUniform4f(loc_color, 0.7, 0.6, 0.85, 1.0)

        mat_scale = matriz_escala(0.211 * fechamento_cortina, 1.0, 0.01)
        
        mat_rotation_y = matriz_rotacao_y(angulo_tira)
        
        mat_translation = matriz_translacao(0.6 + (px - 0.6) * fechamento_cortina, 0.0, 0.0)

        mat_transform = multiplica_matriz(mat_rotation_y, mat_scale)
        mat_transform = multiplica_matriz(mat_translation, mat_transform)
        
        mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
        mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
        mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
        mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
        mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
        mat_transform = multiplica_matriz(mat_cena, mat_transform)


        glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
        glDrawArrays(GL_TRIANGLES, inicio_cubo, quantos_vertices_cubo)

    # APOIO CORTINA 1 --------------------------------
    mat_translation = matriz_translacao(0.7, 0.5, 0.04)
    mat_rotation_y = matriz_rotacao_y(90)
    mat_scale = matriz_escala(0.18, 0.07, 0.07)

    mat_transform = multiplica_matriz(mat_rotation_y, mat_scale)
    mat_transform = multiplica_matriz(mat_translation, mat_transform)
    
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)


    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, 0.3, 0.2, 0.1, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_cubo, quantos_vertices_cubo)

    # APOIO CORTINA 2 --------------------------------
    mat_translation = matriz_translacao(-0.7, 0.5, 0.04)
    mat_rotation_y = matriz_rotacao_y(90)
    mat_scale = matriz_escala(0.18, 0.07, 0.07)

    mat_transform = multiplica_matriz(mat_rotation_y, mat_scale)
    mat_transform = multiplica_matriz(mat_translation, mat_transform)
    
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)


    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, 0.3, 0.2, 0.1, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_cubo, quantos_vertices_cubo)


'''
Função de criação e posicionamento da janela.
Parametros:
    - pos_x, pos_y, pos_z: posição da janela no espaço
    - escala_janela: escala da janela como um todo
    - angulo_x, angulo_y, angulo_z: ângulo de rotação da janela em torno dos eixos x, y e z
'''
def desenha_janela(pos_x, pos_y, pos_z, escala_janela, angulo_x, angulo_y, angulo_z, mat_cena):

    loc = glGetUniformLocation(program, "mat_transformation")

    # Transformação global
    centro_x = pos_x
    centro_y = pos_y
    centro_z = pos_z

    angulo_janela_x
    angulo_janela_y
    angulo_janela_z


    mat_rot_global_x = matriz_rotacao_x(angulo_janela_x + angulo_x)
    mat_rot_global_y = matriz_rotacao_y(angulo_janela_y + angulo_y)
    mat_rot_global_z = matriz_rotacao_z(angulo_janela_z + angulo_z)
    mat_scale_global = matriz_escala(escala_janela, escala_janela, escala_janela)
    mat_translation_global = matriz_translacao(centro_x, centro_y, centro_z)

    # VIDRO ------------------------------------
    mat_translation = matriz_translacao(0.0, 0.0, 0.0)
    mat_scale = matriz_escala(0.8, 0.8, 0.09)

    mat_transform = multiplica_matriz(mat_translation, mat_scale)
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)


    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, 0.5, 0.7, 0.9, 0.3)  # transparente
    glDrawArrays(GL_TRIANGLES, inicio_cubo, quantos_vertices_cubo)

    # MOLDURA SUPERIOR -----------------------------------------------------
    mat_translation = matriz_translacao(0.0, 0.45, 0.0)
    mat_scale = matriz_escala(1, 0.1, 0.15)

    mat_transform = multiplica_matriz(mat_translation, mat_scale)
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)

    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, 0.4, 0.2, 0.1, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_cubo, quantos_vertices_cubo)

    # MOLDURA INFERIOR ---------------------------------------------
    mat_translation = matriz_translacao(0.0, -0.45, 0.0)
    mat_scale = matriz_escala(1, 0.1, 0.15)

    mat_transform = multiplica_matriz(mat_translation, mat_scale)
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)

    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, 0.4, 0.2, 0.1, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_cubo, quantos_vertices_cubo)

    # MOLDURA ESQUERDA ------------------------------------------------
    mat_translation = matriz_translacao(-0.45, 0.0, 0.0)
    mat_scale = matriz_escala(0.1, 0.9, 0.15)

    mat_transform = multiplica_matriz(mat_translation, mat_scale)
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)

    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, 0.4, 0.2, 0.1, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_cubo, quantos_vertices_cubo)

    # MOLDURA DIREITA -------------------------------------------------
    mat_translation = matriz_translacao(0.45, 0.0, 0.0)
    mat_scale = matriz_escala(0.1, 0.9, 0.15)

    mat_transform = multiplica_matriz(mat_translation, mat_scale)
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)

    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, 0.4, 0.2, 0.1, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_cubo, quantos_vertices_cubo)

    # MOLDURA INTERNA VERTICAL -------------------------------------------------
    mat_translation = matriz_translacao(0.0, 0.0, 0.0)
    mat_scale = matriz_escala(0.1, 0.9, 0.12)

    mat_transform = multiplica_matriz(mat_translation, mat_scale)
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)

    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, 0.4, 0.2, 0.1, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_cubo, quantos_vertices_cubo)

    # MOLDURA INTERNA HORIZONTAL ---------------------------------------------
    mat_translation = matriz_translacao(0.0, 0.0, 0.0)
    mat_scale = matriz_escala(1, 0.1, 0.12)

    mat_transform = multiplica_matriz(mat_translation, mat_scale)
    mat_transform = multiplica_matriz(mat_rot_global_x, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_y, mat_transform)
    mat_transform = multiplica_matriz(mat_rot_global_z, mat_transform)
    mat_transform = multiplica_matriz(mat_scale_global, mat_transform)
    mat_transform = multiplica_matriz(mat_translation_global, mat_transform)
    mat_transform = multiplica_matriz(mat_cena, mat_transform)

    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)
    glUniform4f(loc_color, 0.4, 0.2, 0.1, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_cubo, quantos_vertices_cubo)

---
## Janela final

### Exibição da janela

In [184]:
glfw.show_window(window)

### Loop principal da janela

Lembrando alguns controles adicionais para facilitar a visualização: 

* [↑], [↓], [→], [←], [Z], [X]: para controlar a rotação da cena
* [R]: para reiniciar a cena.
* [P]: para mostrar a malha poligonal.

In [185]:
glEnable(GL_DEPTH_TEST)

while not glfw.window_should_close(window):

    glfw.poll_events()

    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)
    glClearColor(1.0, 1.0, 1.0, 1.0)

    if malha_poligonal == True:
        glPolygonMode(GL_FRONT_AND_BACK, GL_LINE)
    else:
        glPolygonMode(GL_FRONT_AND_BACK, GL_FILL)

    if ventilador_ligado:
        angulo_laminas += 3.0

    mat_rot_cena_x = matriz_rotacao_x(angulo_cena_x)
    mat_rot_cena_y = matriz_rotacao_y(angulo_cena_y)
    mat_rot_cena_z = matriz_rotacao_z(angulo_cena_z)

    mat_cena = multiplica_matriz(mat_rot_cena_y, mat_rot_cena_x)
    mat_cena = multiplica_matriz(mat_rot_cena_z, mat_cena)

    desenha_ventilador(0.45, -0.36, 0.45, 0.56, 0, 30, 0, mat_cena)
    desenha_sofa(0.0, -0.25, -0.455, 0.94, 0, 180, 0, mat_cena)
    desenha_aspirador(-0.4, -0.5, 0, 0.5, -90, -90, 0, mat_cena)
    desenha_ambiente(0.0, -0.25, 0.0, 0.52, 0, 0, 0, mat_cena)
    desenha_tv(0.0, 0.1, 0.557, 1.0, 0, 0, 0, mat_cena)
    desenha_cortina(0.52, 0.1, 0.0, 0.5, 0, 90, 0, mat_cena)
    desenha_janela(0.604, 0.1, 0.0, 0.5, 0, 90, 0, mat_cena)
    

    glfw.swap_buffers(window)

glfw.terminate()

